In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
# Used to create token ids, encode data, and decode tokens
class Processor:
    def __init__(self):
        self.encodings = {}
        self.decodings = {}

    # Read data to create token ids
    def ingest(self, data=str):
        raw_chars = list(data)
        unique_chars = sorted(list(set(raw_chars)))

        # Assign each char to a token id
        for token_id, u_char in enumerate(unique_chars):
            self.encodings[u_char]    = token_id
            self.decodings[token_id] = u_char

        # Assign vocab size
        self.vocab_size = len(unique_chars)

    # Encode characters to token ids
    def encode(self, data=str):
        raw_chars = list(data)
        tokens = [self.encodings[raw_char] for raw_char in raw_chars]
        return tokens
    
    # Decode tokens into characters
    def decode(self, data=list):
        decoded_tokens = [self.decodings[token_id] for token_id in data]
        decoded_string = "".join(decoded_tokens)
        return decoded_string

In [3]:
processor = Processor()

# Read and ingest data
with open("Data/tiny-shakespeare.txt", 'r') as f:
  data = f.read()
processor.ingest(data)
tokenized_data_list = processor.encode(data)

# Convert into tensor
tokenized_data = torch.tensor(tokenized_data_list, dtype=torch.long)

# Get vocab size
vocab_size = processor.vocab_size

In [ ]:
# Creates instance of one model
# All hyperparameters and training are done inside this object
class Model():
    def __init__(self, vocab_size, B=32, T=64, C=128, H=128, lr=1e-3):
        # Define hyper parameters
        self.B = B   # Batch size
        self.T = T   # Sequence length or Block size
        self.C = C   # Embedding dimension
        self.H = H   # Matches C as we only are implementing one head
        self.lr = lr # Learning rate
        self.vocab_size = vocab_size

        # Define matrices
        self.embedding_matrix = torch.randn(vocab_size, C, device = device) / torch.sqrt(self.C)  # Holds embedding vectors for each token (Vocab_Size x C)
        self.W_q = torch.randn(H, C, device = device) / torch.sqrt(self.C)    # Holds the Query weights (What we look for given input)
        self.W_k = torch.randn(H, C, device = device) / torch.sqrt(self.C)    # Holds the Key weights (What the input holds/represents or has to offer)
        self.W_v = torch.randn(H, C, device = device) / torch.sqrt(self.C)    # Holds the Value weights (Content that should be passed forward)
        ### Since our Head size is the same as our Embedding dimension, we can use the embedding matrix as our lm_head matrix
        ### I chose (H x C) dimensions because torches nn.Linear stores the parameter dimensions backwards like above
        ### This allows for similar computation with transposing the weights

    # Create (B, T, C) matrix of randomly selected tokens from given data
    def build_train_batch(self, data):
        indices = torch.randint(len(data) - self.T - 1, (self.B,), device = device)
        x_ids = torch.stack([data[ix:ix+self.T] for ix in indices]) # (B, T)
        y = torch.stack([data[ix+1:ix+self.T+1] for ix in indices]) # (B, T)

        # Replace token ids with their embedding vectors
        X = self.embedding_matrix[x_ids]    # (B, T, C)

        return x_ids, X, y
        
    # Calculate Q and K to get our pre-softmax attention matrix (A)
    def get_affinities(self, X):
        # Get our Query and Key matrices
        self.Q = X @ self.W_q.T    # (B, T, C) @ (C, H) --> (B, T, H)
        self.K = X @ self.W_k.T    # (B, T, C) @ (C, H) --> (B, T, H)
        ### This moves from the embedding dimension to our head size dimension
        ### In our case the head size is equal to the embedding dimension, so not much change happens here

        self.A = self.Q @ self.K.transpose(-2, -1)     # (B, T, H) @ (B, H, T) --> (B, T, T)
        self.A = self.A / (torch.sqrt(self.H))         # Scaling to prevent crazy value growth
        ### I use .tranpose here to manually swap dimension -2 and -1, or T and H, to allow correct matrix multiplication
        

    # Given we have our affinities, A, we now turn it to a lower triangle and softmax
    # We do so by setting the upper triangle to -inf
    # This ensures the softmax excludes future tokens, preventing a token from looking into the 'future'
    def softmax_attention(self):
        # Generate a lower triangle of ones - Then set 0's to -infinity
        tril = torch.tril(torch.ones(self.T, self.T, device = device))
        A_shifted = self.A - self.A.max(dim=-1, keepdim=True).values  # Shifts A by subtracting max of each row to each element (Prevents overflow cases)
        A_masked = A_shifted.masked_fill(tril == 0, float('-inf'))  # --> (B, T, T) with only lower triangles maintained

        # Exponentiate all elements
        exp_vals = torch.exp(A_masked)
        exp_row_sums = exp_vals.sum(dim=-1, keepdim=True)    # Get the sums of each row (dim=-1 or dim=2)

        # Calculate softmaxes by dividing each exponentiated value by its rows sum of exponentiates 
        self.S = exp_vals / exp_row_sums   
        ### S is still --> (B, T, T) 

    # This is where our 'learning' is retrieved. 
    # Using the softmaxed attention values, S, we take a weighted average of the 'content'
    def value_aggregation(self, X):
        # Get our Value matrix
        self.V = X @ self.W_v.T      # (B, T, C) @ (C, H) --> (B, T, H)

        # Obtain our output before undoing projection
        self.O = self.S @ self.V   # (B, T, T) @ (B, T, H) --> (B, T, H)
        
        # Bring our output back to C dim and get logits
        self.Z = self.O @ self.embedding_matrix.T     # (B, T, H) @ (C, Vocab size) --> (B, T, Vocab Size)
        ### The only reason I used embedding matrix here is because H == C
        ### When the head size does NOT equal C, I must add a lm_head matrix of dimenion (Vocab size, H)

    # Softmax for our probabilities of next token
    def logits_to_p(self):
        # Exponentiate all elements
        exp_vals = torch.exp(self.Z)
        exp_row_sums = exp_vals.sum(dim=-1, keepdim=True)    # Get the sums of each row (dim=-1 or dim=2)

        # Calculate softmaxes by dividing each exponentiated value by its rows sum of exponentiates 
        self.P = exp_vals / exp_row_sums     # (B, T, Vocab_size)
        ### Now we have our probabilities for the next token of each token in each batch

    # Perform one forward pass to calculate predicted output
    def forward_pass(self, X, y):      # X is expected (B, T, C)
        self.get_affinities(X)      # Q @ K.T  
        self.softmax_attention()    # Softmax(A)
        self.value_aggregation(X)   # O = S @ V --> Z = O * lm_head
        self.logits_to_p()          # Softmax(Z)

    
    ##### WEIGHT UPDATING #####
    # Must be called first - GZ is defined here and used in other gradients
    def gradient_W_v(self, X, y):
        # X - (B, T, C)
        # A - (B, T, T)
        # P - (B, T, Vocab size)
        # Y - (B, T)
        # W_lm or embedding matrix - (Vocab size, C)
        # Gradient, or G, of A = Derivative of Loss wrt. A
        # Our softmaxes are based off our logits, Z = O @ W_lm
        # We know GZ = P - Y
        # GO = (P - Y) @ W_lm                          - GO --> (B, T, Vocab size) @ (Vocab size, C) --> (B, T, C)
        # O = A @ V so linear derivative property says - GV = A.T @ GO   --> (B, T, T) @ (B, T, C) --> (B, T, C)
        # V = X @ W_V                                  - GW_V = X.T @ GV --> (B, C, T) @ (B, T, C) --> (B, C, C)
        # So GW_V = X.T ( A.T @ [( P - Y) @ W_lm])     - (B, C, T) @ [ (B, T, T) @ [ (B, T, Vocab Size) @ (Vocab Size, C) ] ]
        #                                              - (B, C, T) @ [ (B, T, T) @ [ (B, T, C)]]
        #                                              - (B, C, T) @ [ (B, T, C)]
        #                                              - (B, C, C)      * Consistent *
        self.GZ = self.P.clone()
        B_indices = torch.arange(self.B, device=device).view(-1, 1)    # (B, 1)
        T_indices = torch.arange(self.T, device=device).view(1, -1)    # (1, T)
        self.GZ[B_indices, T_indices, y] -= 1                # (B, T, Vocab Size)

        GO = self.GZ @ self.embedding_matrix     # (B, T, C)
        self.GV = self.A.transpose(-2, -1) @ GO  # (B, T, C)
        GW_V = X.transpose(-2, -1) @ self.GV     # (B, C, C)

        # We need sum of gradients across batches
        GW_V = GW_V.sum(dim=0)              # (C, H)
        return GW_V.T                       # (H, C) for updating W_V which is also (H, C)

    def gradient_W_qk(self, X, y):
        # X                                 - (B, T, C)
        # GZ = P - Y                        - (B, T, Vocab size)
        # W_lm or embedding matrix          - (Vocab size, C)
        # GO = GZ @ W_lm                    - (B, T, C) - Only 'C' because C equals H
        # V                                 - (T, H)
        # GA = GO @ V^T                     - (B, T, T) 
        # GS = A * (GA - rowsum(A * GA))    - (B, T, T)
        ### K and Q                         - (B, T, H)
        # W_k and W_q                       - (H, C)
        # Q = X @ W^T_q                     - (B, T, H)
        # K = X @ W^T_k                     - (B, T, H)
        # Since S = Q @ K^T, the transpose derivative property says:
        # GQ = GS @ K                       - (B, T, H)
        # GK = (GS)^T @ Q                   - (B, T, H)
        # GW_Q = (GQ)^T @ X                 - (B, H, C) or (B, C, C) in our case
        # GW_K = (GK)^T @ X                 - (B, H, C) or (B, C, C) in our case
        
        GO = self.GZ @ self.embedding_matrix    # (Vocab size, C)
        GA = GO @ self.V.transpose(-2, -1)      # (B, T, T)
        A_GA = self.A * GA                      # (B, T, T) - Element wise multiplication
        rowsums = A_GA.sum(dim=-1, keepdim=True)# (B, T, 1)
        GS = self.A * (GA - rowsums)            # (B, T, T)
        GS = GS / torch.sqrt(self.H)            # Scaling to prevent crazy value growth

        # Derive gradients for Q and K weights
        self.GQ = GS @ self.K                        # (B, T, H)
        self.GK = GS.transpose(-2, -1) @ self.Q      # (B, T, H)

        GW_Q = self.GQ.transpose(-2, -1) @ X         # (B, H, C) or (B, C, C)
        GW_K = self.GK.transpose(-2, -1) @ X         # (B, H, C) or (B, C, C)
        
        GW_Q = GW_Q.sum(dim=0)                  # (H, C)
        GW_K = GW_K.sum(dim=0)                  # (H, C)

        return GW_Q, GW_K
    
    def gradient_W_e(self, x_ids, X, y):
        # O                 - (B, T, H) or (B, T, C)
        # Z = O @ W^T_e     - (B, T, Vocab size)
        # GZ = P - Y        - (B, T, Vocab Size)
        ## Output gradient ##
        # GW_e = (GZ)^T @ O - (B, Vocab size, C)
        ## Input gradient ##
        # GX = GQ @ W_Q + GK @ W_K + GV @ W_V   - (B, T, C)
        
        # Output gradient
        GW_E_output = self.GZ.transpose(-2, -1) @ self.O   # (B, Vocab size, C)
        GW_E_output = GW_E_output.sum(dim=0)                      # (Vocab size, C)

        # Input gradient
        GX = (self.GQ * self.W_q) + (self.GK * self.W_k) + (self.GV * self.W_v) # (B, T, C)
        GW_E_input = torch.zeros_line(self.embedding_matrix)    # (Vocab size, C)
        GW_E_input.index_add_(0, x_ids.view(-1), GX.view(-1, self.C))

        return GW_E_output + GW_E_input

    def back_pass(self, x_ids, X, y):
        # Get gradients
        GW_V = self.gradient_W_v(X, y)
        GW_Q, GW_K = self.gradient_W_qk(X, y)
        GW_E = self.gradient_W_e(x_ids, X, y)

        # Update weights
        self.W_v -= self.lr * GW_V
        self.W_q -= self.lr * GW_Q
        self.W_k -= self.lr * GW_K 
        self.embedding_matrix -= self.lr * GW_E

    ##### TRAINING LOOP #####
    def train(self, data, max_iter=1e5):
        for i in range(max_iter):
            # Get random batches
            x_ids, X, y = self.build_train_batch(data)

            self.forward_pass(X, y)
            self.back_pass(x_ids, X, y)




In [52]:
model = Model(vocab_size)

x, y = model.build_train_batch(tokenized_data)
torch.set_printoptions(profile="full")
print(f"{x[:2]}")

tensor([[40, 53, 50, 42,  1, 61, 47, 58, 46,  1, 63, 53, 59,  0, 32, 53,  1, 45,
         47, 60, 43,  1, 63, 53, 59,  1, 53, 60, 43, 56,  1, 39, 58,  1, 58, 46,
         47, 57,  1, 44, 47, 56, 57, 58,  1, 43, 52, 41, 53, 59, 52, 58, 43, 56,
          6,  0, 33, 52, 50, 43, 57, 57,  1, 63],
        [50, 42,  1, 19, 39, 59, 52, 58,  1, 41, 53, 51, 51, 43, 52, 42, 57,  1,
         46, 47, 51,  1, 58, 53,  1, 63, 53, 59, 56,  1, 51, 39, 48, 43, 57, 58,
         63,  8,  0,  0, 23, 21, 26, 19,  1, 30, 21, 15, 20, 13, 30, 16,  1, 21,
         21, 10,  0, 35, 46, 39, 58,  1, 57, 39]])


In [51]:
tril = torch.tril(torch.ones(5, 5))

print(tril)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
